### Run your existing Bike Shop setup to load its tables into PostgreSQL, then import the required libraries, connect to the database and verify that bicycles, sales and servicerecords are available

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text

user = 'pgadmin'
password = 'geheim'
host = 'localhost'
port = '5432'
database = 'postgres'

db_connection_url = f'postgresql://{user}:{password}@{host}:{port}/{database}'
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT current_database() AS database_name,                                 --Verbindung zur erwarteten PostgreSQL-Datenbank kontrollieren
           (SELECT COUNT(*) FROM bicycles) AS bicycles_count,                   --Fahrradbestand prüfen; erwartet sind im Quelldatensatz 2000 Zeilen
           (SELECT COUNT(*) FROM sales) AS sales_count,                         --Vorhandensein der Bike-Shop-Verkaufstabelle prüfen
           (SELECT COUNT(*) FROM servicerecords) AS services_count;             --Vorhandensein der Bike-Shop-Servicetabelle prüfen
""", con=engine)

engine.dispose()
display(df_spickap01)

,database_name,bicycles_count,sales_count,services_count
0,postgres,2000,684,7500


### EXTRACT — Retrieve each bicycle sale with its bicycle ID, make, model, sale date and sale year. Display the year as an integer

In [2]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT b.bicycleid, b.make, b.model, s.saledate,                    --Fahrrad und Verkaufsdatum anzeigen
           CAST(EXTRACT(YEAR FROM s.saledate) AS INTEGER) AS sale_year  --Jahr aus dem Datum lesen und als Ganzzahl ausgeben
    FROM bicycles AS b                                                  --Fahrradtabelle verwenden
    INNER JOIN sales AS s ON b.bicycleid = s.bicycleid                  --Verkaufsdatum über die Fahrrad-ID ergänzen
    ORDER BY s.saledate, b.bicycleid;                                   --Chronologisch und bei gleichem Datum nach Fahrrad-ID sortieren
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,saledate,sale_year
0,C9PGTJQX18,Cannondale,SuperSix,2019-08-29,2019
1,C8QUWFU87V,Santa Cruz,Bronson,2019-09-06,2019
2,CJSMB9CVGE,Bianchi,Infinito,2019-09-13,2019
3,CP64NZ2VJ0,Giant,Talon,2019-09-14,2019
4,C2WFIK6WEP,Merida,Scultura,2019-09-15,2019
...,...,...,...,...,...
679,C6P05HHSPM,Giant,TCR,2024-08-11,2024
680,CMMCJADVB1,Cannondale,Trail,2024-08-14,2024
681,CSX9QRNFZE,Scott,Aspect,2024-08-14,2024
682,CWE0XO1VM0,Giant,TCR,2024-08-16,2024


### EXTRACT + GROUP BY — Determine the average cost per year for services described as "Gear shifting adjustment". Display year and average_service_costs, rounded to two decimal places and sorted by average_service_costs in descending order

In [3]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT CAST(EXTRACT(YEAR FROM servicedate) AS INTEGER) AS year,       --Jahr des Services auslesen und als Ganzzahl ausgeben
           ROUND(CAST(AVG(cost) AS NUMERIC), 2) AS average_service_costs  --Durchschnittliche Kosten pro Jahr auf zwei Stellen runden
    FROM servicerecords                                                   --Serviceeinträge des Bike Shops verwenden
    WHERE servicedescription = 'Gear shifting adjustment'                 --Nur diese Servicebeschreibung berücksichtigen
    GROUP BY CAST(EXTRACT(YEAR FROM servicedate) AS INTEGER)              --Alle passenden Services desselben Jahres zusammenfassen
    ORDER BY average_service_costs DESC;                                  --Höchsten durchschnittlichen Servicepreis zuerst anzeigen
""", con=engine)

engine.dispose()
display(df_spickap01)

,year,average_service_costs
0,2019,25.83
1,2021,25.71
2,2020,25.40
3,2022,25.30
4,2023,25.07
5,2024,24.59


### CAST — Retrieve each bicycle ID and its year, both as the original number and as text. Name the additional column year_text

In [5]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, year,                                       --ID und ursprüngliches Baujahr auswählen
           CAST(year AS TEXT) AS year_text                        --Baujahr nur für die Ausgabe in Text umwandeln
    FROM bicycles;                                                --Der gespeicherte Datentyp wird dadurch nicht verändert
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,year,year_text
0,C7ZGWFLBUV,2019,2019
1,C875HZ1AG6,2021,2021
2,CDONEWSWFP,2021,2021
3,C2UQZP803C,2023,2023
4,C2WFIK6WEP,2020,2020
...,...,...,...
1995,CLH919EIKV,2020,2020
1996,CP3F48BE9Y,2022,2022
1997,CHBXQ3SSNU,2019,2019
1998,CP5CTME2ZD,2021,2021


### :: — Retrieve each bicycle ID and its year as text. Use PostgreSQL shorthand casting and name the result year_text

In [6]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                                             --Fahrrad-ID auswählen
           year::TEXT AS year_text                                --PostgreSQL-Kurzform für CAST(year AS TEXT)
    FROM bicycles;                                                --Nur die Abfrageausgabe umwandeln, nicht die Originalspalte
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,year_text
0,C7ZGWFLBUV,2019
1,C875HZ1AG6,2021
2,CDONEWSWFP,2021
3,C2UQZP803C,2023
4,C2WFIK6WEP,2020
...,...,...
1995,CLH919EIKV,2020
1996,CP3F48BE9Y,2022
1997,CHBXQ3SSNU,2019
1998,CP5CTME2ZD,2021


### ROUND + calculation — Retrieve each bicycle ID, make, model and price, and calculate a price reduced by 10%. Round discounted_price to two decimal places

In [7]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price,                               --Originaldaten und ursprünglichen Preis anzeigen
           ROUND(CAST(price * 0.90 AS NUMERIC), 2) AS discounted_price  --Zehn Prozent Rabatt berechnen und auf zwei Stellen runden
    FROM bicycles;                                                      --Berechnung nur ausgeben; gespeicherte Preise bleiben unverändert
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price,discounted_price
0,C7ZGWFLBUV,Trek,Domane,5066.94,4560.25
1,C875HZ1AG6,Merida,Reacto,5529.38,4976.44
2,CDONEWSWFP,Merida,Big Nine,1518.33,1366.50
3,C2UQZP803C,Trek,Marlin,5038.99,4535.09
4,C2WFIK6WEP,Merida,Scultura,1548.19,1393.37
...,...,...,...,...,...
1995,CLH919EIKV,Trek,Emonda,2376.02,2138.42
1996,CP3F48BE9Y,Merida,Scultura,2833.36,2550.02
1997,CHBXQ3SSNU,Merida,Scultura,1629.91,1466.92
1998,CP5CTME2ZD,Trek,Marlin,6176.55,5558.90


### COALESCE — Retrieve each bicycle ID, make, model and status. Display "Unknown" when the status is missing and name that output column status_display

In [8]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model,                                --Fahrraddaten auswählen
           COALESCE(status, 'Unknown') AS status_display          --Ersten nicht-NULL-Wert ausgeben; leeren Text nicht ersetzen
    FROM bicycles;                                                --Nur die Ausgabe anpassen; Originaldaten bleiben unverändert
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,status_display
0,C7ZGWFLBUV,Trek,Domane,In Maintenance
1,C875HZ1AG6,Merida,Reacto,Sold
2,CDONEWSWFP,Merida,Big Nine,In Maintenance
3,C2UQZP803C,Trek,Marlin,In Maintenance
4,C2WFIK6WEP,Merida,Scultura,Sold
...,...,...,...,...
1995,CLH919EIKV,Trek,Emonda,Available
1996,CP3F48BE9Y,Merida,Scultura,Available
1997,CHBXQ3SSNU,Merida,Scultura,In Maintenance
1998,CP5CTME2ZD,Trek,Marlin,In Maintenance


### CASE — Retrieve each bicycle ID and price, and classify the price as "Unknown" if missing, "Budget" below 2000, "Mid-range" from 2000 to 4000 inclusive, or "Premium" above 4000. Name the new column price_category

In [9]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, price,                                      --Fahrrad-ID und ursprünglichen Preis anzeigen
           CASE                                                   --Fallunterscheidung beginnen; erster passender WHEN-Zweig entscheidet
               WHEN price IS NULL THEN 'Unknown'                  --Fehlenden Preis ausdrücklich behandeln
               WHEN price < 2000 THEN 'Budget'                    --Preise unter 2000 zuordnen
               WHEN price <= 4000 THEN 'Mid-range'                --Nach dem vorigen Zweig bleiben hier Preise von 2000 bis 4000
               ELSE 'Premium'                                     --Übrige Preise über 4000 zuordnen
           END AS price_category                                  --Fallunterscheidung beenden und Ergebnisspalte benennen
    FROM bicycles;                                                --Keine Preiswerte in der Datenbank ändern
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,price,price_category
0,C7ZGWFLBUV,5066.94,Premium
1,C875HZ1AG6,5529.38,Premium
2,CDONEWSWFP,1518.33,Budget
3,C2UQZP803C,5038.99,Premium
4,C2WFIK6WEP,1548.19,Budget
...,...,...,...
1995,CLH919EIKV,2376.02,Mid-range
1996,CP3F48BE9Y,2833.36,Mid-range
1997,CHBXQ3SSNU,1629.91,Budget
1998,CP5CTME2ZD,6176.55,Premium


### || — Retrieve each bicycle ID and combine make and model into one space-separated column named bicycle_name.

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                                             --Fahrrad-ID auswählen
           make || ' ' || model AS bicycle_name                   --Marke, Leerzeichen und Modell zu einem Text verbinden
    FROM bicycles;                                                --Bei NULL in einem Bestandteil ist die gesamte Verkettung NULL
""", con=engine)

engine.dispose()
display(df_spickap01)

### UPPER — Retrieve each bicycle ID, its original make and the make in uppercase. Name the additional column make_upper

In [10]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make,                                       --Fahrrad-ID und Originalmarke auswählen
           UPPER(make) AS make_upper                              --Marke in Grossbuchstaben ausgeben
    FROM bicycles;                                                --Originalmarke bleibt gespeichert wie zuvor
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,make_upper
0,C7ZGWFLBUV,Trek,TREK
1,C875HZ1AG6,Merida,MERIDA
2,CDONEWSWFP,Merida,MERIDA
3,C2UQZP803C,Trek,TREK
4,C2WFIK6WEP,Merida,MERIDA
...,...,...,...
1995,CLH919EIKV,Trek,TREK
1996,CP3F48BE9Y,Merida,MERIDA
1997,CHBXQ3SSNU,Merida,MERIDA
1998,CP5CTME2ZD,Trek,TREK


### LOWER — Retrieve each bicycle ID, its original make and the make in lowercase. Name the additional column make_lower

In [11]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make,                                       --Fahrrad-ID und Originalmarke auswählen
           LOWER(make) AS make_lower                              --Marke in Kleinbuchstaben ausgeben
    FROM bicycles;                                                --Gross-/Kleinschreibung nur in der Ausgabe ändern
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,make_lower
0,C7ZGWFLBUV,Trek,trek
1,C875HZ1AG6,Merida,merida
2,CDONEWSWFP,Merida,merida
3,C2UQZP803C,Trek,trek
4,C2WFIK6WEP,Merida,merida
...,...,...,...
1995,CLH919EIKV,Trek,trek
1996,CP3F48BE9Y,Merida,merida
1997,CHBXQ3SSNU,Merida,merida
1998,CP5CTME2ZD,Trek,trek


### LENGTH — Retrieve each bicycle ID, model and the number of characters in the model name. Name the additional column model_length and sort by it in descending order

In [12]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, model,                                      --Fahrrad-ID und Modell auswählen
           LENGTH(model) AS model_length                          --Zeichen inklusive Leerzeichen zählen; NULL bleibt NULL
    FROM bicycles                                                 --Fahrradtabelle verwenden
    ORDER BY model_length DESC NULLS LAST, bicycleid;             --Längste Modellnamen zuerst; bei Gleichstand nach ID sortieren
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,model,model_length
0,C15T1IUY8R,Rockhopper,10
1,C19I6367PV,Rockhopper,10
2,C1AP0CNR2L,Rockhopper,10
3,C2HX6YP3UN,Rockhopper,10
4,C34OXLI9KY,Rockhopper,10
...,...,...,...
1995,CXSPK20YP0,R5,2
1996,CXW9GFQT40,R5,2
1997,CZENR3UUJU,R5,2
1998,CZMECIKHMX,R5,2


### REPLACE — Retrieve each bicycle ID and status, and show a version in which "In Maintenance" is replaced by "Maintenance". Name the additional column status_display

In [13]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, status,                                                   --Fahrrad-ID und Originalstatus anzeigen
           REPLACE(status, 'In Maintenance', 'Maintenance') AS status_display   --Exakten Text ersetzen; Gross-/Kleinschreibung beachten
    FROM bicycles;                                                              --Nur Ausgabetext ändern, nicht den gespeicherten Status
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,status,status_display
0,C7ZGWFLBUV,In Maintenance,Maintenance
1,C875HZ1AG6,Sold,Sold
2,CDONEWSWFP,In Maintenance,Maintenance
3,C2UQZP803C,In Maintenance,Maintenance
4,C2WFIK6WEP,Sold,Sold
...,...,...,...
1995,CLH919EIKV,Available,Available
1996,CP3F48BE9Y,Available,Available
1997,CHBXQ3SSNU,In Maintenance,Maintenance
1998,CP5CTME2ZD,In Maintenance,Maintenance


### SUBSTRING — Retrieve each bicycle ID, model and the first three characters of its model name. Name the additional column model_prefix

In [14]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, model,                                      --Fahrrad-ID und Originalmodell auswählen
           SUBSTRING(model FROM 1 FOR 3) AS model_prefix          --Ab Position 1 drei Zeichen entnehmen; SQL beginnt bei 1
    FROM bicycles;                                                --Kurze Modellnamen liefern entsprechend weniger Zeichen
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,model,model_prefix
0,C7ZGWFLBUV,Domane,Dom
1,C875HZ1AG6,Reacto,Rea
2,CDONEWSWFP,Big Nine,Big
3,C2UQZP803C,Marlin,Mar
4,C2WFIK6WEP,Scultura,Scu
...,...,...,...
1995,CLH919EIKV,Emonda,Emo
1996,CP3F48BE9Y,Scultura,Scu
1997,CHBXQ3SSNU,Scultura,Scu
1998,CP5CTME2ZD,Marlin,Mar


### AND + OR + parentheses — Retrieve bicycle ID, make, model and price for bicycles made by Bianchi or Trek, but only if their price is greater than 3000

In [15]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                          --Gewünschte Fahrradspalten auswählen
    FROM bicycles                                                 --Fahrradtabelle verwenden
    WHERE (make = 'Bianchi' OR make = 'Trek')                     --Klammern fassen die beiden Markenalternativen zusammen
      AND price > 3000;                                           --Preisbedingung gilt für beide Marken; AND bindet sonst stärker als OR
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C7ZGWFLBUV,Trek,Domane,5066.94
1,C2UQZP803C,Trek,Marlin,5038.99
2,C2ZEK0YPD9,Bianchi,Oltre,7382.81
3,CE5XORIKH7,Trek,Emonda,7600.96
4,CB46L6E7V3,Trek,Emonda,4396.75
...,...,...,...,...
267,CB4PK8PLSN,Bianchi,Oltre,7765.04
268,C1V12YL0CG,Trek,Domane,7497.23
269,CFFTG1MNW0,Bianchi,Aria,5939.92
270,C6DHR3STN7,Trek,Marlin,6624.90


### Subquery: overall average — Retrieve bicycle ID, make, model and price for bicycles priced above the overall average bicycle price. Sort by price in descending order

In [16]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                          --Fahrraddaten der äusseren Abfrage auswählen
    FROM bicycles                                                 --Fahrradtabelle verwenden
    WHERE price > (SELECT AVG(price) FROM bicycles)               --Innere Abfrage liefert einen Vergleichswert: den Gesamtdurchschnitt
    ORDER BY price DESC, bicycleid;                               --Höchste Preise zuerst; bei gleichem Preis nach ID sortieren
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C187CGZK39,Trek,Marlin,7995.90
1,CENB0TLQKE,Bianchi,Oltre,7989.74
2,CJSLUUOM7Z,Santa Cruz,Hightower,7984.31
3,CLQCGBXCAG,Merida,Scultura,7972.88
4,CTVKH7S3PZ,Trek,Marlin,7972.58
...,...,...,...,...
984,CKA8RS2OQG,Trek,Emonda,4400.88
985,C70A8JPBP6,Scott,Aspect,4399.97
986,CB46L6E7V3,Trek,Emonda,4396.75
987,CM6ANT1O7T,Cervelo,Soloist,4392.99


### IN with subquery — Retrieve bicycle ID, make, model and price for bicycles that appear in at least one sales record

In [17]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                          --Fahrraddaten auswählen
    FROM bicycles                                                 --Jedes Fahrrad der äusseren Tabelle betrachten
    WHERE bicycleid IN (                                          --ID muss in der Ergebnismenge der Unterabfrage enthalten sein
        SELECT bicycleid                                          --Unterabfrage liefert genau eine Spalte
        FROM sales                                                --Fahrrad-IDs aus den Verkaufsdatensätzen verwenden
    );                                                            --Mehrere Verkäufe vervielfachen die Zeile der äusseren Tabelle nicht
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C875HZ1AG6,Merida,Reacto,5529.38
1,C2WFIK6WEP,Merida,Scultura,1548.19
2,CGPCXD8911,Scott,Aspect,1715.48
3,C3U71M5BP5,Specialized,Rockhopper,7904.79
4,CLD9BSTE3H,Specialized,Rockhopper,5712.21
...,...,...,...,...
679,CALG7KLJHE,Trek,Domane,6629.37
680,C78URTU7TV,Specialized,Rockhopper,7375.79
681,C1V12YL0CG,Trek,Domane,7497.23
682,C6OURRJ88X,Specialized,Rockhopper,4208.90


### NOT IN with subquery — Retrieve bicycle ID, make, model and price for bicycles that do not appear in any sales record. Exclude missing IDs from the subquery

In [19]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                          --Fahrraddaten auswählen
    FROM bicycles                                                 --Jedes Fahrrad der äusseren Tabelle betrachten
    WHERE bicycleid NOT IN (                                      --Nur IDs behalten, die in der Unterabfrage fehlen
        SELECT bicycleid                                          --Verkaufte Fahrrad-IDs auswählen
        FROM sales                                                --Verkaufstabelle verwenden
        WHERE bicycleid IS NOT NULL                               --NULL ausschliessen; ein NULL in NOT IN kann sonst Treffer verhindern
    );                                                            --Unterabfrage liefert ausschliesslich vorhandene IDs
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C7ZGWFLBUV,Trek,Domane,5066.94
1,CDONEWSWFP,Merida,Big Nine,1518.33
2,C2UQZP803C,Trek,Marlin,5038.99
3,C4RQCO25I9,Cube,Reaction,3339.27
4,C9LPTOS9KE,Cannondale,Synapse,5399.61
...,...,...,...,...
1311,CLH919EIKV,Trek,Emonda,2376.02
1312,CP3F48BE9Y,Merida,Scultura,2833.36
1313,CHBXQ3SSNU,Merida,Scultura,1629.91
1314,CP5CTME2ZD,Trek,Marlin,6176.55


### EXISTS — Retrieve bicycle ID, make, model and price for bicycles with at least one sales record. Use EXISTS

In [20]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT b.bicycleid, b.make, b.model, b.price                  --Fahrraddaten der äusseren Tabelle auswählen
    FROM bicycles AS b                                            --b ist der Kurzname der äusseren Fahrradtabelle
    WHERE EXISTS (                                                --Prüfen, ob mindestens ein passender Datensatz existiert
        SELECT 1                                                  --1 ist nur ein Platzhalter; EXISTS zählt keine Zeilen
        FROM sales AS s                                           --Passenden Verkaufsdatensatz suchen
        WHERE s.bicycleid = b.bicycleid                           --Unterabfrage mit dem aktuell betrachteten Fahrrad verknüpfen
    );                                                            --Auch bei mehreren Verkäufen jedes Fahrrad nur einmal auswählen
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C875HZ1AG6,Merida,Reacto,5529.38
1,C2WFIK6WEP,Merida,Scultura,1548.19
2,CGPCXD8911,Scott,Aspect,1715.48
3,C3U71M5BP5,Specialized,Rockhopper,7904.79
4,CLD9BSTE3H,Specialized,Rockhopper,5712.21
...,...,...,...,...
679,CALG7KLJHE,Trek,Domane,6629.37
680,C78URTU7TV,Specialized,Rockhopper,7375.79
681,C1V12YL0CG,Trek,Domane,7497.23
682,C6OURRJ88X,Specialized,Rockhopper,4208.90


### NOT EXISTS — Retrieve bicycle ID, make, model and price for bicycles without any sales record. Use NOT EXISTS

In [21]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT b.bicycleid, b.make, b.model, b.price                  --Fahrraddaten auswählen
    FROM bicycles AS b                                            --Jedes Fahrrad der äusseren Tabelle betrachten
    WHERE NOT EXISTS (                                            --Nur Fahrräder ohne passenden Verkaufsdatensatz behalten
        SELECT 1                                                  --Entscheidend ist die Existenz eines Treffers, nicht der Wert 1
        FROM sales AS s                                           --Verkaufstabelle durchsuchen
        WHERE s.bicycleid = b.bicycleid                           --Nach Verkäufen des aktuellen Fahrrads suchen
    );                                                            --NULL-Werte in anderen Verkaufszeilen verhindern diese Prüfung nicht
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C7ZGWFLBUV,Trek,Domane,5066.94
1,CDONEWSWFP,Merida,Big Nine,1518.33
2,C2UQZP803C,Trek,Marlin,5038.99
3,C4RQCO25I9,Cube,Reaction,3339.27
4,C9LPTOS9KE,Cannondale,Synapse,5399.61
...,...,...,...,...
1311,CLH919EIKV,Trek,Emonda,2376.02
1312,CP3F48BE9Y,Merida,Scultura,2833.36
1313,CHBXQ3SSNU,Merida,Scultura,1629.91
1314,CP5CTME2ZD,Trek,Marlin,6176.55


### ANY — Retrieve bicycle ID, make, model and price for bicycles that cost more than at least one Bianchi bicycle with a known price

In [22]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                          --Fahrraddaten auswählen
    FROM bicycles                                                 --Alle Marken der äusseren Tabelle berücksichtigen
    WHERE price > ANY (                                           --Mindestens einer der Vergleiche muss zutreffen
        SELECT price                                              --Vergleichspreise auswählen
        FROM bicycles                                             --Bianchi-Preise aus derselben Tabelle lesen
        WHERE make = 'Bianchi' AND price IS NOT NULL              --Nur bekannte Bianchi-Preise als Vergleich verwenden
    );                                                            --Bei leerer Unterabfrage ist > ANY FALSE
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C7ZGWFLBUV,Trek,Domane,5066.94
1,C875HZ1AG6,Merida,Reacto,5529.38
2,CDONEWSWFP,Merida,Big Nine,1518.33
3,C2UQZP803C,Trek,Marlin,5038.99
4,C2WFIK6WEP,Merida,Scultura,1548.19
...,...,...,...,...
1959,CLH919EIKV,Trek,Emonda,2376.02
1960,CP3F48BE9Y,Merida,Scultura,2833.36
1961,CHBXQ3SSNU,Merida,Scultura,1629.91
1962,CP5CTME2ZD,Trek,Marlin,6176.55


### ALL — Retrieve bicycle ID, make, model and price for bicycles that cost more than every Bianchi bicycle with a known price

In [23]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                          --Fahrraddaten auswählen
    FROM bicycles                                                 --Alle Marken der äusseren Tabelle berücksichtigen
    WHERE price > ALL (                                           --Jeder Vergleich mit einem zurückgegebenen Preis muss zutreffen
        SELECT price                                              --Vergleichspreise auswählen
        FROM bicycles                                             --Bianchi-Preise aus derselben Tabelle lesen
        WHERE make = 'Bianchi' AND price IS NOT NULL              --NULL aus der Vergleichsmenge ausschliessen
    );                                                            --Bei leerer Unterabfrage ist > ALL TRUE; hier sind Bianchi-Preise vorhanden
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C187CGZK39,Trek,Marlin,7995.9


### AVG OVER — Retrieve every bicycle ID, make, model and price, and show the overall average price in an additional column named overall_average_price. Keep every bicycle row

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price,                                          --Einzelne Fahrradzeilen beibehalten
           ROUND(CAST(AVG(price) OVER () AS NUMERIC), 2) AS overall_average_price  --Gesamtdurchschnitt als zusätzliche Spalte in jeder Zeile anzeigen
    FROM bicycles;                                                                 --OVER () fasst die Zeilen im Gegensatz zu GROUP BY nicht zusammen
""", con=engine)

engine.dispose()
display(df_spickap01)

### PARTITION BY — Retrieve every bicycle ID, make, model and price, and show the average price of its make as make_average_price. Keep every bicycle row and round the average to two decimal places

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price,                                                        --Alle einzelnen Fahrräder beibehalten
           ROUND(CAST(AVG(price) OVER (PARTITION BY make) AS NUMERIC), 2) AS make_average_price  --Durchschnitt innerhalb derselben Marke berechnen und runden
    FROM bicycles                                                                                --PARTITION BY bildet Berechnungsgruppen ohne Zeilen zusammenzufassen
    ORDER BY make, bicycleid;                                                                    --Fahrräder nach Marke und ID anzeigen
""", con=engine)

engine.dispose()
display(df_spickap01)

### ROW_NUMBER — Number all bicycles with a known price from most to least expensive. Display bicycleid, make, model, price and row_number. Use bicycle ID in ascending order to break price ties

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price,                                        --Fahrraddaten auswählen
           ROW_NUMBER() OVER (ORDER BY price DESC, bicycleid ASC) AS row_number  --Eindeutige fortlaufende Nummer ab 1 vergeben; ID löst Preisgleichstände auf
    FROM bicycles                                                                --Fahrradtabelle verwenden
    WHERE price IS NOT NULL                                                      --Nur Fahrräder mit bekanntem Preis nummerieren
    ORDER BY row_number;                                                         --Ergebnismenge ausdrücklich nach der berechneten Nummer sortieren
""", con=engine)

engine.dispose()
display(df_spickap01)

### RANK — Rank all bicycles with a known price from most to least expensive. Display bicycleid, make, model, price and price_rank. Equal prices must receive the same rank, with gaps after ties

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price,                         --Fahrraddaten auswählen
           RANK() OVER (ORDER BY price DESC) AS price_rank        --Gleiche Preise erhalten denselben Rang; danach entstehen Ranglücken
    FROM bicycles                                                 --Fahrradtabelle verwenden
    WHERE price IS NOT NULL                                       --Fehlende Preise ausschliessen
    ORDER BY price_rank, bicycleid;                               --Rangfolge anzeigen; ID sortiert Gleichstände, verändert aber nicht den Rang
""", con=engine)

engine.dispose()
display(df_spickap01)

### UNION — Combine the bicycle IDs, makes and models of Bianchi bicycles with those of bicycles priced above 3000. Remove duplicate result rows

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model                                 --Erste Ergebnismenge: Bianchi-Fahrräder
    FROM bicycles                                                 --Fahrradtabelle verwenden
    WHERE make = 'Bianchi'                                        --Auf Marke Bianchi filtern
    UNION                                                         --Ergebnismengen untereinander verbinden; identische vollständige Zeilen entfernen
    SELECT bicycleid, make, model                                 --Zweite Abfrage braucht gleich viele Spalten mit kompatiblen Datentypen
    FROM bicycles                                                 --Erneut aus der Fahrradtabelle lesen
    WHERE price > 3000                                            --Fahrräder über 3000 auswählen
    ORDER BY bicycleid;                                           --Gesamtes vereinigtes Ergebnis sortieren
""", con=engine)

engine.dispose()
display(df_spickap01)

### UNION ALL — Combine the bicycle IDs, makes and models of Bianchi bicycles with those of bicycles priced above 3000. Keep duplicate result rows when a bicycle meets both conditions

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model                                 --Erste Ergebnismenge auswählen
    FROM bicycles                                                 --Fahrradtabelle verwenden
    WHERE make = 'Bianchi'                                        --Bianchi-Fahrräder auswählen
    UNION ALL                                                     --Ergebnisse anhängen, ohne doppelte Zeilen zu entfernen
    SELECT bicycleid, make, model                                 --Spaltenanzahl, Reihenfolge und Datentypen müssen zusammenpassen
    FROM bicycles                                                 --Zweite Ergebnismenge aus derselben Tabelle auswählen
    WHERE price > 3000                                            --Fahrräder über 3000 auswählen; teure Bianchi-Fahrräder erscheinen zweimal
    ORDER BY bicycleid;                                           --Gesamtes Ergebnis nach Fahrrad-ID sortieren
""", con=engine)

engine.dispose()
display(df_spickap01)

## Alle wichtigen Regex

### Regex ~ and ^ — Retrieve bicycle ID, make, model and price for bicycles whose make starts with an uppercase B. Use a case-sensitive regular expression

### Regex können in SELECT stehen und dann True oder False ausgeben

In [25]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query(r"""
    SELECT bicycleid, make, model, framenr,                             --Originalwerte zum Vergleichen anzeigen
           make ~ 'Trek' AS matches_case,                               --~: Treffer mit Beachtung der Gross-/Kleinschreibung
           make ~* 'trek' AS matches_ignore_case,                       --~*: Treffer ohne Beachtung der Gross-/Kleinschreibung
           make !~ 'Trek' AS excludes_case,                             --!~: Kein Treffer mit Beachtung der Gross-/Kleinschreibung
           make !~* 'trek' AS excludes_ignore_case,                     --!~*: Kein Treffer unabhängig von der Gross-/Kleinschreibung
           model ~ '^D' AS starts_with_d,                               --^: Text beginnt mit D
           model ~ 'e$' AS ends_with_e,                                 --$: Text endet mit e
           make ~ '^Trek$' AS exactly_trek,                             --^...$: Gesamter Text muss genau Trek entsprechen
           model ~ 'a.e' AS contains_a_any_e,                           --.: Genau ein beliebiges Zeichen zwischen a und e
           model ~* '[aeiou]' AS contains_vowel,                        --[...]: Mindestens eines der aufgeführten Zeichen
           framenr ~ '[0-9]' AS contains_digit,                         --[0-9]: Eine Ziffer aus dem Bereich 0 bis 9
           framenr ~ '[^0-9]' AS contains_non_digit,                    --[^...]: Ein Zeichen ausserhalb der angegebenen Auswahl
           framenr ~ '\d' AS digit_shortcut,                            --\d: Eine Ziffer; Kurzform für die Ziffernklasse
           model ~ '\s' AS contains_whitespace,                         --\s: Ein Leerraumzeichen, beispielsweise ein Leerzeichen
           model ~ '^\w+$' AS only_word_characters,                     --\w: Buchstabe, Ziffer oder Unterstrich; +: Mindestens eines
           model ~ '^D.*e$' AS starts_d_ends_e,                         --.*: Null oder mehr beliebige Zeichen zwischen D und e
           framenr ~ '^[A-Z]+$' AS only_uppercase_letters,              --[A-Z]+: Mindestens ein Grossbuchstabe; sonst keine Zeichen
           model ~ '^Big\s?Nine$' AS optional_space,                    --?: Vorheriges Element null- oder einmal; BigNine oder Big Nine
           bicycleid ~ '^[A-Z0-9]{10}$' AS id_has_ten_chars,            --{10}: Genau zehn Grossbuchstaben oder Ziffern
           framenr ~ '\d{2,4}' AS two_to_four_digits,                   --{2,4}: Teiltreffer mit zwei bis vier aufeinanderfolgenden Ziffern
           framenr ~ '\d{3,}' AS at_least_three_digits,                 --{3,}: Mindestens drei aufeinanderfolgende Ziffern
           make ~ '^(Trek|Merida)$' AS selected_brands,                 --(...): Gruppierung; |: Trek ODER Merida als vollständiger Wert
           model ~ '\.' AS contains_literal_dot,                        --\.: Einen tatsächlichen Punkt suchen
           model ~ '\mNine\M' AS contains_word_nine                     --\m und \M: Wortanfang und Wortende; Nine als vollständiges Wort
    FROM bicycles                                                       --Fahrradtabelle verwenden
    ORDER BY bicycleid;                                                 --Ergebnisse nach Fahrrad-ID sortieren
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,framenr,matches_case,matches_ignore_case,excludes_case,excludes_ignore_case,starts_with_d,ends_with_e,...,only_word_characters,starts_d_ends_e,only_uppercase_letters,optional_space,id_has_ten_chars,two_to_four_digits,at_least_three_digits,selected_brands,contains_literal_dot,contains_word_nine
0,C02MOC0YLP,Cube,Reaction,XMD5040,False,False,True,True,False,False,...,True,False,False,False,True,True,True,False,False,False
1,C037S02BZX,Cannondale,SuperSix,JRN 233,False,False,True,True,False,False,...,True,False,False,False,True,True,True,False,False,False
2,C03X2W8T7W,Cannondale,Synapse,950 OPV,False,False,True,True,False,True,...,True,False,False,False,True,True,True,False,False,False
3,C04CCUTU6Z,Scott,Spark,72Y TB8,False,False,True,True,False,False,...,True,False,False,False,True,True,False,False,False,False
4,C04U0GF99X,Merida,Big Nine,0019,False,False,True,True,False,True,...,False,False,False,True,True,True,True,True,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,CZXS1CHZTV,Cervelo,Caledonia,DRU-3873,False,False,True,True,False,False,...,True,False,False,False,True,True,True,False,False,False
1996,CZY9JAU92V,Cervelo,Soloist,456 4340,False,False,True,True,False,False,...,True,False,False,False,True,True,True,False,False,False
1997,CZYTYT5C77,Scott,Addict,386M,False,False,True,True,False,False,...,True,False,False,False,True,True,True,False,False,False
1998,CZZEBAGXVC,Scott,Aspect,NG2 T9O,False,False,True,True,False,False,...,True,False,False,False,True,False,False,False,False,False


### Regex als WHERE-Bedingung

In [26]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query(r"""
    SELECT bicycleid, make, model, framenr                                      --Ausgewählte Fahrradspalten anzeigen
    FROM bicycles                                                               --Fahrradtabelle verwenden
    WHERE make ~ '^(Trek|Merida)$'                                              --~: Grossschreibung beachten; ^/$: Ganzer Text; (...)/|: Alternativen
      AND make ~* '^(trek|merida)$'                                             --~*: Gross-/Kleinschreibung ignorieren
      AND make !~ '^Scott$'                                                     --!~: Muster darf mit dieser Schreibweise nicht zutreffen
      AND make !~* '^giant$'                                                    --!~*: Muster darf unabhängig von der Schreibweise nicht zutreffen
      AND model ~ '^[A-Z].*$'                                                   --[A-Z]: Grossbuchstabe am Anfang; .*: Null oder mehr beliebige Zeichen
      AND model ~ '[aeiou]'                                                     --[...]: Mindestens einen der aufgeführten Vokale enthalten
      AND model ~ '\w+'                                                         --\w: Wortzeichen; +: Mindestens ein aufeinanderfolgendes Wortzeichen
      AND model !~ '\d'                                                         --\d: Ziffer; hier Modelle mit Ziffern ausschliessen
      AND model !~ '\.'                                                         --\.: Tatsächlicher Punkt; hier Modelle mit Punkt ausschliessen
      AND bicycleid ~ '^[A-Z0-9]{10}$'                                          --{10}: Genau zehn Grossbuchstaben oder Ziffern
      AND framenr ~ '[^0-9]'                                                    --[^0-9]: Mindestens ein Zeichen, das keine Ziffer ist
      AND framenr ~ '\d{2,4}'                                                   --{2,4}: Teiltreffer mit zwei bis vier aufeinanderfolgenden Ziffern
      AND framenr ~ '\d{3,}'                                                    --{3,}: Mindestens drei aufeinanderfolgende Ziffern
      AND (
          framenr ~ '^[A-Z]{3}-?\d+$'                                           --?: Bindestrich optional; z. B. ABC123 oder ABC-123
          OR framenr ~ '^[A-Z]\d+\s\d[A-Z]{2}$'                                 --\s: Ein Leerraumzeichen; z. B. A15 0LX
      )                                                                         --OR: Eines der beiden Rahmennummernformate genügt
    ORDER BY bicycleid;                                                         --Treffer nach Fahrrad-ID sortieren
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,framenr
0,C06Z017RMC,Merida,Scultura,NEH-2263
1,C1HFOMEBDK,Merida,Scultura,IOH-141
2,C1U71ESAXN,Trek,Emonda,HRN-0839
3,C2Q38KWLVE,Trek,Emonda,SLF-4020
4,C3NN3FVLYA,Merida,Scultura,MRF-1927
5,C3S621WSZS,Trek,Emonda,RJL-3182
6,C4J88XJYQN,Trek,Emonda,QYL-3693
7,C4TZPXHOPX,Trek,Emonda,QVR-8920
8,C55CD1WVKR,Merida,Scultura,KOG-1764
9,C5P7ZABRBY,Merida,Big Nine,IBA-0612


### Regex ~ '^B' — Retrieve the bicycle ID, make, model and price of all bicycles whose make starts with an uppercase B

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                          --Fahrrad-ID, Marke, Modell und Preis auswählen
    FROM bicycles                                                 --Fahrradtabelle verwenden
    WHERE make ~ '^B';                                            --^ markiert den Textanfang; ~ berücksichtigt Gross-/Kleinschreibung
""", con=engine)

engine.dispose()
display(df_spickap01)

### Regex ~* — Retrieve bicycle ID, make, model and price for bicycles whose make starts with B or b. Use a case-insensitive regular expression

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                          --Fahrrad-ID, Marke, Modell und Preis auswählen
    FROM bicycles                                                 --Fahrradtabelle verwenden
    WHERE make ~* '^b';                                           --~* ignoriert Gross-/Kleinschreibung; ^ verlangt den Textanfang
""", con=engine)

engine.dispose()
display(df_spickap01)

### Regex $ — Retrieve bicycle ID, make, model and price for bicycles whose make ends with "chi". Use a regular expression

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                          --Fahrrad-ID, Marke, Modell und Preis auswählen
    FROM bicycles                                                 --Fahrradtabelle verwenden
    WHERE make ~ 'chi$';                                          --$ verlangt das Textende; chi muss am Ende der Marke stehen
""", con=engine)

engine.dispose()
display(df_spickap01)

### Regex ^...$ — Retrieve bicycle ID, make, model and price for bicycles whose make is exactly "Bianchi". Use a regular expression

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                          --Fahrrad-ID, Marke, Modell und Preis auswählen
    FROM bicycles                                                 --Fahrradtabelle verwenden
    WHERE make ~ '^Bianchi$';                                     --^ und $ begrenzen den gesamten Text; keine zusätzlichen Zeichen erlaubt
""", con=engine)

engine.dispose()
display(df_spickap01)

### Regex . — Retrieve bicycle ID, make, model and price for bicycles whose make starts with B, ends with i and has exactly five characters between them

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                          --Fahrrad-ID, Marke, Modell und Preis auswählen
    FROM bicycles                                                 --Fahrradtabelle verwenden
    WHERE make ~ '^B.....i$';                                     --Jeder Punkt steht für genau ein Zeichen; fünf Punkte zwischen B und i
""", con=engine)

engine.dispose()
display(df_spickap01)

### Regex character class — Retrieve bicycle ID, make, model and price for bicycles whose make starts with an uppercase B or T. Use a character class

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                          --Fahrrad-ID, Marke, Modell und Preis auswählen
    FROM bicycles                                                 --Fahrradtabelle verwenden
    WHERE make ~ '^[BT]';                                         --[BT] erlaubt genau ein Zeichen: B oder T; ^ verlangt den Textanfang
""", con=engine)

engine.dispose()
display(df_spickap01)

### Regex [0-9] — Retrieve bicycle ID, make, model and price for bicycles whose model name contains at least one digit

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                          --Fahrrad-ID, Marke, Modell und Preis auswählen
    FROM bicycles                                                 --Fahrradtabelle verwenden
    WHERE model ~ '[0-9]';                                        --Eine Ziffer an beliebiger Stelle genügt; ohne Anker wird irgendwo gesucht
""", con=engine)

engine.dispose()
display(df_spickap01)

### Regex [^0-9] — Retrieve bicycle ID, make, model and price for bicycles whose model name contains at least one character that is not a digit

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                          --Fahrrad-ID, Marke, Modell und Preis auswählen
    FROM bicycles                                                 --Fahrradtabelle verwenden
    WHERE model ~ '[^0-9]';                                       --^ innerhalb der Zeichenklasse bedeutet Ausschluss; auch Leerzeichen zählen
""", con=engine)

engine.dispose()
display(df_spickap01)

### Regex * — Retrieve bicycle ID, make, model and price for bicycles whose model name starts with "Dom" and ends with "e", allowing any number of characters in between

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                          --Fahrrad-ID, Marke, Modell und Preis auswählen
    FROM bicycles                                                 --Fahrradtabelle verwenden
    WHERE model ~ '^Dom.*e$';                                     --.* erlaubt null oder beliebig viele Zeichen zwischen Dom und e
""", con=engine)

engine.dispose()
display(df_spickap01)

### Regex + — Retrieve bicycle ID, make, model and price for bicycles whose model name consists entirely of one or more digits

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                          --Fahrrad-ID, Marke, Modell und Preis auswählen
    FROM bicycles                                                 --Fahrradtabelle verwenden
    WHERE model ~ '^[0-9]+$';                                     --+ verlangt mindestens eine Ziffer; ^ und $ schliessen andere Zeichen aus
""", con=engine)

engine.dispose()
display(df_spickap01)

### Regex ? — Retrieve bicycle ID, make, model and price for bicycles whose model name is exactly "Domane" or "Domanes". Use an optional character

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                          --Fahrrad-ID, Marke, Modell und Preis auswählen
    FROM bicycles                                                 --Fahrradtabelle verwenden
    WHERE model ~ '^Domanes?$';                                   --? macht das unmittelbar vorhergehende s optional; Domane und Domanes passen
""", con=engine)

engine.dispose()
display(df_spickap01)

### Regex {n} — Retrieve bicycle ID, make, model and price for bicycles whose ID starts with C followed by exactly nine uppercase letters or digits

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                          --Fahrrad-ID, Marke, Modell und Preis auswählen
    FROM bicycles                                                 --Fahrradtabelle verwenden
    WHERE bicycleid ~ '^C[A-Z0-9]{9}$';                           --{9} verlangt genau neun Zeichen aus der Klasse nach dem führenden C
""", con=engine)

engine.dispose()
display(df_spickap01)

### Regex {n,m} — Retrieve bicycle ID, make, model and price for bicycles whose model name consists entirely of two to four digits

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                          --Fahrrad-ID, Marke, Modell und Preis auswählen
    FROM bicycles                                                 --Fahrradtabelle verwenden
    WHERE model ~ '^[0-9]{2,4}$';                                 --{2,4} erlaubt mindestens zwei und höchstens vier Ziffern im gesamten Text
""", con=engine)

engine.dispose()
display(df_spickap01)

### Regex alternatives — Retrieve bicycle ID, make, model and price for bicycles whose make is exactly Bianchi or Trek. Use grouped alternatives

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                          --Fahrrad-ID, Marke, Modell und Preis auswählen
    FROM bicycles                                                 --Fahrradtabelle verwenden
    WHERE make ~ '^(Bianchi|Trek)$';                              --Klammern gruppieren Alternativen; | bedeutet ODER; beide Anker gelten für beide Marken
""", con=engine)

engine.dispose()
display(df_spickap01)

### Regex literal dot — Retrieve bicycle ID, make, model and price for bicycles whose model name contains a literal dot. Use a regular expression

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                          --Fahrrad-ID, Marke, Modell und Preis auswählen
    FROM bicycles                                                 --Fahrradtabelle verwenden
    WHERE model ~ '[.]';                                          --[.] sucht einen tatsächlichen Punkt; ohne passende Modellnamen ist das Ergebnis leer
""", con=engine)

engine.dispose()
display(df_spickap01)

### INSERT INTO — Create an empty temporary bicycle table and insert the existing sample bicycle C02MOC0YLP: Cube Reaction, year 2020, price 4823.86, status Sold. Display the inserted row

In [ ]:
engine = create_engine(db_connection_url)

with engine.begin() as connection:
    connection.execute(text("""
    CREATE TEMP TABLE spick_bicycles (                                        --Leere temporäre Übungstabelle mit bekannten Bike-Shop-Spalten erstellen
        bicycleid TEXT,                                                       --Fahrrad-ID als Text speichern
        make VARCHAR(100),                                                    --Marke als Text mit maximal 100 Zeichen speichern
        model VARCHAR(100),                                                   --Modell als Text mit maximal 100 Zeichen speichern
        year INTEGER,                                                         --Baujahr als Ganzzahl speichern
        price NUMERIC(10, 2),                                                 --Preis mit insgesamt zehn Stellen, davon zwei Nachkommastellen
        status TEXT DEFAULT 'Available'                                       --Standardstatus für ausgelassene Werte oder DEFAULT festlegen
    ) ON COMMIT DROP;                                                         --Übungstabelle am Transaktionsende automatisch entfernen
    """))

    connection.execute(text("""
    INSERT INTO spick_bicycles (bicycleid, make, model, year, price, status)  --Zielspalten ausdrücklich in der Reihenfolge der Werte angeben
    VALUES ('C02MOC0YLP', 'Cube', 'Reaction', 2020, 4823.86, 'Sold');         --Einen geprüften Datensatz aus der Quelldatei einfügen; Texte in einfachen Anführungszeichen
    """))

    df_spickap01 = pd.read_sql_query("""
    SELECT *                                                                  --Inhalt der Übungstabelle nach der Änderung anzeigen
    FROM spick_bicycles                                                       --Ausschliesslich die temporäre Kopie abfragen
    ORDER BY bicycleid;                                                       --Ergebnis nach Fahrrad-ID sortieren
    """, con=connection)

engine.dispose()
display(df_spickap01)

### INSERT multiple rows — Create an empty temporary bicycle table and insert both existing samples in one INSERT: C02MOC0YLP (Cube, Reaction, 2020, 4823.86, Sold) and C037S02BZX (Cannondale, SuperSix, 2023, 2424.03, Sold). Display both rows

In [ ]:
engine = create_engine(db_connection_url)

with engine.begin() as connection:
    connection.execute(text("""
    CREATE TEMP TABLE spick_bicycles (                                        --Leere temporäre Übungstabelle mit bekannten Bike-Shop-Spalten erstellen
        bicycleid TEXT,                                                       --Fahrrad-ID als Text speichern
        make VARCHAR(100),                                                    --Marke als Text mit maximal 100 Zeichen speichern
        model VARCHAR(100),                                                   --Modell als Text mit maximal 100 Zeichen speichern
        year INTEGER,                                                         --Baujahr als Ganzzahl speichern
        price NUMERIC(10, 2),                                                 --Preis mit insgesamt zehn Stellen, davon zwei Nachkommastellen
        status TEXT DEFAULT 'Available'                                       --Standardstatus für ausgelassene Werte oder DEFAULT festlegen
    ) ON COMMIT DROP;                                                         --Übungstabelle am Transaktionsende automatisch entfernen
    """))

    connection.execute(text("""
    INSERT INTO spick_bicycles (bicycleid, make, model, year, price, status)  --Spaltenreihenfolge für beide neuen Zeilen festlegen
    VALUES                                                                    --Mehrere Zeilen mit einer INSERT-Anweisung einfügen
        ('C02MOC0YLP', 'Cube', 'Reaction', 2020, 4823.86, 'Sold'),            --Ersten geprüften Datensatz einfügen; Komma trennt die Zeilen
        ('C037S02BZX', 'Cannondale', 'SuperSix', 2023, 2424.03, 'Sold');      --Zweiten geprüften Datensatz einfügen
    """))

    df_spickap01 = pd.read_sql_query("""
    SELECT *                                                                  --Inhalt der Übungstabelle nach der Änderung anzeigen
    FROM spick_bicycles                                                       --Ausschliesslich die temporäre Kopie abfragen
    ORDER BY bicycleid;                                                       --Ergebnis nach Fahrrad-ID sortieren
    """, con=connection)

engine.dispose()
display(df_spickap01)

### INSERT INTO SELECT — Create an empty temporary bicycle table and copy all Bianchi bicycles into it using INSERT INTO SELECT. Display the copied rows

In [ ]:
engine = create_engine(db_connection_url)

with engine.begin() as connection:
    connection.execute(text("""
    CREATE TEMP TABLE spick_bicycles (                                        --Leere temporäre Übungstabelle mit bekannten Bike-Shop-Spalten erstellen
        bicycleid TEXT,                                                       --Fahrrad-ID als Text speichern
        make VARCHAR(100),                                                    --Marke als Text mit maximal 100 Zeichen speichern
        model VARCHAR(100),                                                   --Modell als Text mit maximal 100 Zeichen speichern
        year INTEGER,                                                         --Baujahr als Ganzzahl speichern
        price NUMERIC(10, 2),                                                 --Preis mit insgesamt zehn Stellen, davon zwei Nachkommastellen
        status TEXT DEFAULT 'Available'                                       --Standardstatus für ausgelassene Werte oder DEFAULT festlegen
    ) ON COMMIT DROP;                                                         --Übungstabelle am Transaktionsende automatisch entfernen
    """))

    connection.execute(text("""
    INSERT INTO spick_bicycles (bicycleid, make, model, year, price, status)  --Bereits bestehende Zieltabelle und Zielspalten festlegen
    SELECT bicycleid, make, model, year, price, status                        --Quellspalten in passender Reihenfolge und mit kompatiblen Datentypen auswählen
    FROM bicycles                                                             --Bestehende Fahrraddaten als Quelle verwenden
    WHERE make = 'Bianchi';                                                   --Nur Bianchi-Fahrräder kopieren; Originaltabelle unverändert lassen
    """))

    df_spickap01 = pd.read_sql_query("""
    SELECT *                                                                  --Inhalt der Übungstabelle nach der Änderung anzeigen
    FROM spick_bicycles                                                       --Ausschliesslich die temporäre Kopie abfragen
    ORDER BY bicycleid;                                                       --Ergebnis nach Fahrrad-ID sortieren
    """, con=connection)

engine.dispose()
display(df_spickap01)

### INSERT DEFAULT — Create an empty temporary bicycle table with default status Available. Insert C02MOC0YLP with its original make, model, year and price, but use DEFAULT for status. Display the row

In [ ]:
engine = create_engine(db_connection_url)

with engine.begin() as connection:
    connection.execute(text("""
    CREATE TEMP TABLE spick_bicycles (                                        --Leere temporäre Übungstabelle mit bekannten Bike-Shop-Spalten erstellen
        bicycleid TEXT,                                                       --Fahrrad-ID als Text speichern
        make VARCHAR(100),                                                    --Marke als Text mit maximal 100 Zeichen speichern
        model VARCHAR(100),                                                   --Modell als Text mit maximal 100 Zeichen speichern
        year INTEGER,                                                         --Baujahr als Ganzzahl speichern
        price NUMERIC(10, 2),                                                 --Preis mit insgesamt zehn Stellen, davon zwei Nachkommastellen
        status TEXT DEFAULT 'Available'                                       --Standardstatus für ausgelassene Werte oder DEFAULT festlegen
    ) ON COMMIT DROP;                                                         --Übungstabelle am Transaktionsende automatisch entfernen
    """))

    connection.execute(text("""
    INSERT INTO spick_bicycles (bicycleid, make, model, year, price, status)  --Zielspalten festlegen
    VALUES ('C02MOC0YLP', 'Cube', 'Reaction', 2020, 4823.86, DEFAULT);        --Für status den definierten Standard Available verwenden statt Sold zu kopieren
    """))

    df_spickap01 = pd.read_sql_query("""
    SELECT *                                                                  --Inhalt der Übungstabelle nach der Änderung anzeigen
    FROM spick_bicycles                                                       --Ausschliesslich die temporäre Kopie abfragen
    ORDER BY bicycleid;                                                       --Ergebnis nach Fahrrad-ID sortieren
    """, con=connection)

engine.dispose()
display(df_spickap01)

### UPDATE SET — Create a temporary copy of bicycles. For bicycle C02MOC0YLP, change status to Available and price to 4500. Display that row

In [ ]:
engine = create_engine(db_connection_url)

with engine.begin() as connection:
    connection.execute(text("""
    CREATE TEMP TABLE spick_bicycles ON COMMIT DROP AS           --Temporäre Übungstabelle erstellen; beim Transaktionsende automatisch löschen
    SELECT bicycleid, make, model, year, framenr, price, status  --Originalspalten übernehmen
    FROM bicycles;                                               --Originaldaten kopieren; sämtliche folgenden Änderungen betreffen nur die Kopie
    """))

    connection.execute(text("""
    UPDATE spick_bicycles                                        --Nur die temporäre Übungstabelle verändern
    SET status = 'Available',                                    --Neuen Status festlegen; mehrere Zuweisungen mit Kommas trennen
        price = 4500                                             --Neuen Preis festlegen
    WHERE bicycleid = 'C02MOC0YLP';                              --Änderung auf die angegebene Fahrrad-ID begrenzen; ohne WHERE würden alle Zeilen geändert
    """))

    df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price, status                 --Geänderten Datensatz kontrollieren
    FROM spick_bicycles                                          --Nur aus der Übungskopie lesen
    WHERE bicycleid = 'C02MOC0YLP';                              --Das bearbeitete Fahrrad anzeigen
    """, con=connection)

engine.dispose()
display(df_spickap01)

### UPDATE with calculation — Create a temporary copy of bicycles. Increase the prices of all Bianchi bicycles by 5 percent and round them to two decimal places. Display the affected bicycles

In [ ]:
engine = create_engine(db_connection_url)

with engine.begin() as connection:
    connection.execute(text("""
    CREATE TEMP TABLE spick_bicycles ON COMMIT DROP AS           --Temporäre Übungstabelle erstellen; beim Transaktionsende automatisch löschen
    SELECT bicycleid, make, model, year, framenr, price, status  --Originalspalten übernehmen
    FROM bicycles;                                               --Originaldaten kopieren; sämtliche folgenden Änderungen betreffen nur die Kopie
    """))

    connection.execute(text("""
    UPDATE spick_bicycles                                        --Preise ausschliesslich in der Übungskopie ändern
    SET price = ROUND(CAST(price * 1.05 AS NUMERIC), 2)          --Neuen Preis aus dem bisherigen Wert berechnen und runden
    WHERE make = 'Bianchi';                                      --Nur Fahrräder der Marke Bianchi verändern
    """))

    df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                         --Neue Preise kontrollieren
    FROM spick_bicycles                                          --Temporäre Kopie abfragen
    WHERE make = 'Bianchi'                                       --Nur betroffene Fahrräder anzeigen
    ORDER BY bicycleid;                                          --Nach ID sortieren
    """, con=connection)

engine.dispose()
display(df_spickap01)

### UPDATE FROM — Create a temporary copy of bicycles. Build a price_updates table with prices 10 percent above the original Bianchi prices, then transfer these prices to the copy using UPDATE FROM. Display the updated Bianchi bicycles

In [ ]:
engine = create_engine(db_connection_url)

with engine.begin() as connection:
    connection.execute(text("""
    CREATE TEMP TABLE spick_bicycles ON COMMIT DROP AS           --Temporäre Übungstabelle erstellen; beim Transaktionsende automatisch löschen
    SELECT bicycleid, make, model, year, framenr, price, status  --Originalspalten übernehmen
    FROM bicycles;                                               --Originaldaten kopieren; sämtliche folgenden Änderungen betreffen nur die Kopie
    """))

    connection.execute(text("""
    CREATE TEMP TABLE price_updates ON COMMIT DROP AS            --Temporäre Quelltabelle für neue Preise anlegen
    SELECT bicycleid,                                            --Fahrrad-ID als Zuordnungsschlüssel übernehmen
           ROUND(CAST(price * 1.10 AS NUMERIC), 2) AS new_price  --Neue Preise aus den Originalpreisen berechnen
    FROM bicycles                                                --Originaldaten nur lesen
    WHERE make = 'Bianchi';                                      --Pro Bianchi-Fahrrad genau einen Quellwert bereitstellen
    """))

    connection.execute(text("""
    UPDATE spick_bicycles AS b                                   --Zieltabelle und Kurznamen festlegen
    SET price = p.new_price                                      --Preis aus der Quelltabelle übernehmen
    FROM price_updates AS p                                      --Tabelle mit neuen Preisen ergänzen
    WHERE b.bicycleid = p.bicycleid;                             --Über ID verbinden; pro Zielzeile sollte höchstens eine Quellzeile passen
    """))

    df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                         --Übernommene Preise anzeigen
    FROM spick_bicycles                                          --Temporäre Zieltabelle abfragen
    WHERE make = 'Bianchi'                                       --Betroffene Marke auswählen
    ORDER BY bicycleid;                                          --Nach Fahrrad-ID sortieren
    """, con=connection)

engine.dispose()
display(df_spickap01)

### UPDATE NULL — Create a temporary copy of bicycles. Set the price of bicycle C02MOC0YLP to NULL and display that row

In [ ]:
engine = create_engine(db_connection_url)

with engine.begin() as connection:
    connection.execute(text("""
    CREATE TEMP TABLE spick_bicycles ON COMMIT DROP AS           --Temporäre Übungstabelle erstellen; beim Transaktionsende automatisch löschen
    SELECT bicycleid, make, model, year, framenr, price, status  --Originalspalten übernehmen
    FROM bicycles;                                               --Originaldaten kopieren; sämtliche folgenden Änderungen betreffen nur die Kopie
    """))

    connection.execute(text("""
    UPDATE spick_bicycles                                        --Nur die Übungskopie ändern
    SET price = NULL                                             --Preis als fehlend speichern; NULL bedeutet nicht 0
    WHERE bicycleid = 'C02MOC0YLP';                              --Änderung auf ein bestimmtes Fahrrad beschränken
    """))

    df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                         --Datensatz mit nun fehlendem Preis anzeigen
    FROM spick_bicycles                                          --Übungskopie abfragen
    WHERE bicycleid = 'C02MOC0YLP';                              --Bearbeitetes Fahrrad auswählen
    """, con=connection)

engine.dispose()
display(df_spickap01)

### DELETE — Create a temporary copy of bicycles. Delete bicycle C02MOC0YLP and verify that no row with this ID remains in the copy

In [ ]:
engine = create_engine(db_connection_url)

with engine.begin() as connection:
    connection.execute(text("""
    CREATE TEMP TABLE spick_bicycles ON COMMIT DROP AS           --Temporäre Übungstabelle erstellen; beim Transaktionsende automatisch löschen
    SELECT bicycleid, make, model, year, framenr, price, status  --Originalspalten übernehmen
    FROM bicycles;                                               --Originaldaten kopieren; sämtliche folgenden Änderungen betreffen nur die Kopie
    """))

    connection.execute(text("""
    DELETE FROM spick_bicycles                                   --Zeile aus der Übungskopie löschen; die Tabelle bleibt bestehen
    WHERE bicycleid = 'C02MOC0YLP';                              --Nur diese ID löschen; ohne WHERE würden alle Zeilen gelöscht
    """))

    df_spickap01 = pd.read_sql_query("""
    SELECT COUNT(*) AS remaining_rows                            --Verbleibende Treffer zählen; erwartet wird 0
    FROM spick_bicycles                                          --Nur die Übungstabelle prüfen
    WHERE bicycleid = 'C02MOC0YLP';                              --Nach der gelöschten ID suchen
    """, con=connection)

engine.dispose()
display(df_spickap01)

### CREATE TABLE — Create an empty temporary table named spick_bicycles with bicycleid as text, make and model as VARCHAR(100), year as INTEGER, price as NUMERIC(10,2) and status defaulting to Available. Display its columns

In [ ]:
engine = create_engine(db_connection_url)

with engine.begin() as connection:
    connection.execute(text("""
    CREATE TEMP TABLE spick_bicycles (   --Leere temporäre Übungstabelle mit bekannten Bike-Shop-Spalten erstellen
        bicycleid TEXT,                  --Fahrrad-ID als Text speichern
        make VARCHAR(100),               --Marke als Text mit maximal 100 Zeichen speichern
        model VARCHAR(100),              --Modell als Text mit maximal 100 Zeichen speichern
        year INTEGER,                    --Baujahr als Ganzzahl speichern
        price NUMERIC(10, 2),            --Preis mit insgesamt zehn Stellen, davon zwei Nachkommastellen
        status TEXT DEFAULT 'Available'  --Standardstatus für ausgelassene Werte oder DEFAULT festlegen
    ) ON COMMIT DROP;                    --Übungstabelle am Transaktionsende automatisch entfernen
    """))

    df_spickap01 = pd.read_sql_query("""
    SELECT *                             --Spalten der neu erstellten Tabelle anzeigen
    FROM spick_bicycles;                 --Leere Tabelle liefert Spaltennamen, aber noch keine Datenzeilen
    """, con=connection)

engine.dispose()
display(df_spickap01)

### SELECT INTO — Create and fill a new temporary table named spick_expensive_bicycles with the ID, make, model and price of bicycles priced above 4000. Use SELECT INTO and display the new table

In [ ]:
engine = create_engine(db_connection_url)

with engine.begin() as connection:
    connection.execute(text("""
    SELECT bicycleid, make, model, price  --Gewünschte Quellspalten auswählen
    INTO TEMP spick_expensive_bicycles    --Neue temporäre Tabelle erstellen und mit dem Abfrageergebnis füllen
    FROM bicycles                         --Originaltabelle nur lesen
    WHERE price > 4000;                   --Nur teure Fahrräder übernehmen; spätere Quelländerungen aktualisieren die Kopie nicht
    """))

    df_spickap01 = pd.read_sql_query("""
    SELECT *                              --Alle Spalten der neuen Tabelle anzeigen
    FROM spick_expensive_bicycles         --Neu erstellte Tabelle verwenden
    ORDER BY price DESC, bicycleid;       --Teuerste Fahrräder zuerst; temporäre Tabelle endet mit dieser Verbindung
    """, con=connection)

engine.dispose()
display(df_spickap01)

### ALTER ADD COLUMN — Create a temporary copy of bicycles and add an empty color column with type VARCHAR(30). Display five rows including the new column

In [ ]:
engine = create_engine(db_connection_url)

with engine.begin() as connection:
    connection.execute(text("""
    CREATE TEMP TABLE spick_bicycles ON COMMIT DROP AS           --Temporäre Übungstabelle erstellen; beim Transaktionsende automatisch löschen
    SELECT bicycleid, make, model, year, framenr, price, status  --Originalspalten übernehmen
    FROM bicycles;                                               --Originaldaten kopieren; sämtliche folgenden Änderungen betreffen nur die Kopie
    """))

    connection.execute(text("""
    ALTER TABLE spick_bicycles                                   --Struktur der temporären Tabelle ändern
    ADD COLUMN color VARCHAR(30);                                --Neue Textspalte ergänzen; ohne DEFAULT sind bestehende Zeilen dort NULL
    """))

    df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, color                         --Neue Spalte zusammen mit bestehenden Daten anzeigen
    FROM spick_bicycles                                          --Temporäre Kopie verwenden
    ORDER BY bicycleid                                           --Stabile Reihenfolge festlegen
    LIMIT 5;                                                     --Fünf Beispielzeilen anzeigen
    """, con=connection)

engine.dispose()
display(df_spickap01)

### ALTER COLUMN TYPE — Create a temporary copy of bicycles and change the stored year column to TEXT using an explicit conversion. Display five bicycle IDs, years and their resulting PostgreSQL data type

In [ ]:
engine = create_engine(db_connection_url)

with engine.begin() as connection:
    connection.execute(text("""
    CREATE TEMP TABLE spick_bicycles ON COMMIT DROP AS           --Temporäre Übungstabelle erstellen; beim Transaktionsende automatisch löschen
    SELECT bicycleid, make, model, year, framenr, price, status  --Originalspalten übernehmen
    FROM bicycles;                                               --Originaldaten kopieren; sämtliche folgenden Änderungen betreffen nur die Kopie
    """))

    connection.execute(text("""
    ALTER TABLE spick_bicycles                                   --Struktur der Übungskopie verändern
    ALTER COLUMN year TYPE TEXT USING year::TEXT;                --Gespeicherten Spaltentyp ändern; USING legt die Konvertierung fest
    """))

    df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, year, pg_typeof(year)::TEXT AS year_type   --Werte und tatsächlichen PostgreSQL-Datentyp kontrollieren
    FROM spick_bicycles                                          --Temporäre Kopie verwenden
    ORDER BY bicycleid                                           --Stabile Reihenfolge festlegen
    LIMIT 5;                                                     --Fünf Beispielzeilen anzeigen
    """, con=connection)

engine.dispose()
display(df_spickap01)

### ALTER VARCHAR length — Create a temporary copy of bicycles with model initially stored as VARCHAR(100). Increase the maximum length to 200 characters and display five rows

In [ ]:
engine = create_engine(db_connection_url)

with engine.begin() as connection:
    connection.execute(text("""
    CREATE TEMP TABLE spick_bicycles ON COMMIT DROP AS           --Temporäre Übungstabelle erstellen; beim Transaktionsende automatisch löschen
    SELECT bicycleid, make, model, year, framenr, price, status  --Originalspalten übernehmen
    FROM bicycles;                                               --Originaldaten kopieren; sämtliche folgenden Änderungen betreffen nur die Kopie
    """))

    connection.execute(text("""
    ALTER TABLE spick_bicycles                                   --Kopie zunächst auf einen begrenzten Texttyp setzen
    ALTER COLUMN model TYPE VARCHAR(100);                        --Aus dem ursprünglichen TEXT wird Text mit maximal 100 Zeichen
    """))

    connection.execute(text("""
    ALTER TABLE spick_bicycles                                   --Struktur der Übungskopie erneut ändern
    ALTER COLUMN model TYPE VARCHAR(200);                        --Erlaubte Textlänge auf 200 Zeichen erhöhen; bestehende Werte bleiben erhalten
    """))

    df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model                                --Bestehende Werte nach der Strukturänderung anzeigen
    FROM spick_bicycles                                          --Übungskopie abfragen
    ORDER BY bicycleid                                           --Nach ID sortieren
    LIMIT 5;                                                     --Fünf Beispielzeilen anzeigen
    """, con=connection)

engine.dispose()
display(df_spickap01)

### ALTER DROP COLUMN — Create a temporary copy of bicycles, add a color column and then remove that column. Display five rows of the remaining table

In [ ]:
engine = create_engine(db_connection_url)

with engine.begin() as connection:
    connection.execute(text("""
    CREATE TEMP TABLE spick_bicycles ON COMMIT DROP AS           --Temporäre Übungstabelle erstellen; beim Transaktionsende automatisch löschen
    SELECT bicycleid, make, model, year, framenr, price, status  --Originalspalten übernehmen
    FROM bicycles;                                               --Originaldaten kopieren; sämtliche folgenden Änderungen betreffen nur die Kopie
    """))

    connection.execute(text("""
    ALTER TABLE spick_bicycles                                   --Übungskopie verändern
    ADD COLUMN color VARCHAR(30);                                --Spalte für die Löschübung zuerst erstellen
    """))

    connection.execute(text("""
    ALTER TABLE spick_bicycles                                   --Tabelle bleibt bestehen
    DROP COLUMN color;                                           --Nur die Spalte color und ihre gespeicherten Werte entfernen
    """))

    df_spickap01 = pd.read_sql_query("""
    SELECT *                                                     --Verbleibende Spalten anzeigen; color fehlt
    FROM spick_bicycles                                          --Übungskopie abfragen
    ORDER BY bicycleid                                           --Nach ID sortieren
    LIMIT 5;                                                     --Fünf Beispielzeilen anzeigen
    """, con=connection)

engine.dispose()
display(df_spickap01)

### DROP TABLE — Create a temporary copy of bicycles, drop the copy and verify that the temporary table no longer exists

In [ ]:
engine = create_engine(db_connection_url)

with engine.begin() as connection:
    connection.execute(text("""
    CREATE TEMP TABLE spick_bicycles ON COMMIT DROP AS                      --Temporäre Übungstabelle erstellen; beim Transaktionsende automatisch löschen
    SELECT bicycleid, make, model, year, framenr, price, status             --Originalspalten übernehmen
    FROM bicycles;                                                          --Originaldaten kopieren; sämtliche folgenden Änderungen betreffen nur die Kopie
    """))

    connection.execute(text("""
    DROP TABLE spick_bicycles;                                              --Übungstabelle vollständig inklusive Struktur und Daten löschen
    """))

    df_spickap01 = pd.read_sql_query("""
    SELECT to_regclass('pg_temp.spick_bicycles') IS NULL AS table_removed;  --TRUE bestätigt das Fehlen der temporären Tabelle; bicycles bleibt unverändert
    """, con=connection)

engine.dispose()
display(df_spickap01)